# Supa Strikers: T5 only (team & role re-classification of existing tracks, Kaggle T4 ×2)

Runs **only the T5 stage** of `kaggle_player_tracking.ipynb` on a `final_tracks_<mode>.parquet` that already
exists: no detection, no tracking. Use it to (re)classify teams and roles quickly.

**Inputs** (Add Input; found anywhere under `/kaggle/input` by name): the tactical dataset
(`BAR-ATH_tactical.mp4`, `BAR-ATH_tactical_atd.txt` / `.xml`, `BAR-ATH_tactical_homography.csv`) and a dataset
with **`final_tracks_full.parquet`** (a previous player-tracking output).
**Settings:** GPU **T4 ×2** (one model copy per GPU), Internet *on* (SigLIP2 weights, ~1.6 GB).

**What it does:** pseudo-labels from the ATD's team membership (no hand labels) → SigLIP2-so400m at 224 px on
whole-body crops → logistic regression over home / away / home GK / away GK / official, checked on held-out
2-minute blocks → one vote per 10 s track segment (5 crops) + the goalkeeper position rule → checks against the
ATD → `final_tracks_<mode>_t5.parquet` (original columns, old labels kept as `team_v1` / `role_v1`).
A progress line with the ETA is printed every minute.

**How to run:** `RUN_MODE = "smoke"` (first 2,000 frames of each half, a few minutes) → check the crop grid and
the ATD agreement → `"full"` → *Save & Run All*.

In [ ]:
import os
RUN_MODE = "smoke"          # "smoke": first SMOKE_FRAMES of each half; "full": both halves
SMOKE_FRAMES = 2000
FPS = 25
PITCH = (105.3, 68.0)
N_VIDEO = 199319
HALVES = [(1, 11185, 90651), (2, 110808, 190197)]   # (half, first, last) tactical video frames (DATA.md)
CALIB_MATCH_GATE = 1.5      # m; a box must sit this close to an ATD player to take its team
DATA_DIR = os.environ.get("SS_DATA_DIR", "/kaggle/input")
WORK = os.environ.get("SS_WORK", "/kaggle/working")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
import sys
if WORK not in sys.path:
    sys.path.insert(0, WORK)


# Strong team & role model (T5): runs after the export, on the final tracks (see the last section)
USE_T5 = True
T5_MODELS = ("google/siglip2-so400m-patch14-224",      # same model as 384 at 224 px: ~3x faster, crops are 40-90 px anyway
             "google/siglip2-so400m-patch14-384", "google/siglip-so400m-patch14-384", "google/siglip-base-patch16-224")
T5_TRACKS = None            # None -> search the inputs for final_tracks_<mode>.parquet; or a path
T5_TRAIN_FRAMES = 1200      # frames across the scored halves for ATD pseudo-labels (sun AND shade)
T5_BLOCK_S = 120            # held-out check: train on even 2-min blocks, test on odd ones
T5_SEG_SECONDS = 10         # tracks are judged in 10 s segments: an ID switch cannot flip a whole track
T5_CROPS_PER_SEG = 5        # crops per segment (evenly spread)
T5_MAX_PER_CLASS = 3000     # training crops per class (home / away have ~11k each: more adds time, not accuracy)
T5_MULTI_GPU = True         # one model copy per GPU (T4 x2), batches embedded in parallel while the video is decoded
T5_MIN_BOX_H = 18           # px; smaller boxes are skipped for appearance
T5_PAD = 0.15               # whole-body crop padding (fraction of the box)
T5_BATCH = 64
T5_REF_CONF = 0.4           # detector "referee" boxes at least this confident become official pseudo-labels
T5_GK_MAX_DIST = 25.0       # m; a segment whose median position is further than this from both goal lines
                            # cannot be a goalkeeper (settles goalkeeper vs referee in similar kits)

In [ ]:
import glob, json, math, shutil, time, gc
import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
print("torch", torch.__version__, "| CUDA devices:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if torch.cuda.device_count() < 2:
    print("NOTE: fewer than 2 GPUs visible: set Accelerator to 'GPU T4 x2' (T5 then runs about half as fast)")

In [ ]:
%%writefile common.py
"""Shared code for the notebook and the GPU detection workers.

Roles, torso crops, colour histograms, batched ResNet18 / SigLIP embeddings and the frozen team
models fitted at calibration. Imported by det_worker.py (one process per GPU) and by the notebook.
"""
import json
import os

import cv2
import numpy as np
import torch
import torch.nn.functional as F

ROLE_PLAYER, ROLE_GK, ROLE_REF = 0, 1, 2
ROLE_NAME = {ROLE_PLAYER: "player", ROLE_GK: "goalkeeper", ROLE_REF: "referee"}
HOME, AWAY, NO_TEAM = 0, 1, -1
TEAM_LETTER = {HOME: "H", AWAY: "A", NO_TEAM: ""}

# Torso band (fraction of box height) and box sanity limits, from DeepTeamClassifier.
TORSO_Y = (0.15, 0.50)
MIN_W, MIN_H = 12, 25
AR_MIN, AR_MAX = 1.0, 5.0       # height / width
COLOR_WEIGHT = 2.0              # weight of the HSV histogram inside the T1 feature


def role_map_from_names(names):
    """{model class id: role}. Ball / pitch classes are ignored.

    A plain COCO model only has 'person': it is treated as player (referees then end up in the
    player pool and show up in the metrics as overflow).
    """
    out = {}
    for k, v in dict(names).items():
        n = str(v).lower().strip()
        if "ball" in n or "pitch" in n:
            continue
        if "goal" in n or "keeper" in n or n == "gk":
            out[int(k)] = ROLE_GK
        elif "ref" in n:
            out[int(k)] = ROLE_REF
        elif "player" in n or n == "person":
            out[int(k)] = ROLE_PLAYER
    return out


def torso_crop(frame, box, min_w=None, min_h=None):
    """BGR torso crop of one box, or None when the box can't be trusted. ``min_w`` / ``min_h``
    loosen the size limits (the detection worker uses 8 x 16 px so far-side assistants get scored)."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = [int(round(float(v))) for v in box]
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
    bw, bh = x2 - x1, y2 - y1
    if bw < (MIN_W if min_w is None else min_w) or bh < (MIN_H if min_h is None else min_h):
        return None
    ar = bh / bw
    if ar < AR_MIN or ar > AR_MAX:
        return None
    ty1, ty2 = y1 + int(bh * TORSO_Y[0]), y1 + int(bh * TORSO_Y[1])
    if ty2 <= ty1:
        return None
    crop = frame[ty1:ty2, x1:x2]
    return crop if crop.size else None


def hsv_hist(crop_bgr):
    """16x8 Hue/Saturation histogram, L2-normalised (128 values). Value is left out on purpose."""
    hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [16, 8], [0, 180, 0, 256]).flatten().astype(np.float32)
    n = np.linalg.norm(hist)
    return hist / n if n > 1e-8 else hist


def letterbox(crop_bgr, size=224):
    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    s = max(h, w)
    top, left = (s - h) // 2, (s - w) // 2
    sq = cv2.copyMakeBorder(rgb, top, s - h - top, left, s - w - left, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return cv2.resize(sq, (size, size), interpolation=cv2.INTER_LINEAR)


def l2n(x):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, 1e-8)


def t1_features(hist, rn):
    return l2n(np.concatenate([COLOR_WEIGHT * hist, rn], axis=1)).astype(np.float32)


class Embedder:
    """Batched ResNet18 (ImageNet) and, optionally, SigLIP embeddings of torso crops."""

    def __init__(self, device=None, use_siglip=True, siglip_name="google/siglip-base-patch16-224"):
        from torchvision.models import resnet18, ResNet18_Weights

        self.device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        self.half = self.device.startswith("cuda")
        m = resnet18(weights=None if os.environ.get("SS_NO_WEIGHTS") else ResNet18_Weights.DEFAULT)
        m.fc = torch.nn.Identity()
        self.resnet = m.to(self.device).eval()
        if self.half:
            self.resnet.half()
        self.mean = torch.tensor([0.485, 0.456, 0.406], device=self.device).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], device=self.device).view(1, 3, 1, 1)
        self.siglip = None
        if use_siglip:
            try:
                from transformers import SiglipVisionModel
                self.siglip = SiglipVisionModel.from_pretrained(siglip_name).to(self.device).eval()
                if self.half:
                    self.siglip.half()
            except Exception as e:  # no internet, old transformers ... -> T2 is skipped
                print("SigLIP unavailable, T2 disabled:", repr(e)[:200])

    @torch.no_grad()
    def embed(self, crops, bs=256):
        """crops: list of BGR arrays -> (resnet (n,512), siglip (n,768) or None)."""
        imgs = [letterbox(c) for c in crops]
        rn_out, sg_out = [], []
        for i in range(0, len(imgs), bs):
            x = torch.from_numpy(np.stack(imgs[i:i + bs])).to(self.device).permute(0, 3, 1, 2).float() / 255.0
            xr = (x - self.mean) / self.std
            if self.half:
                xr = xr.half()
            rn_out.append(F.normalize(self.resnet(xr).float(), dim=1).cpu().numpy())
            if self.siglip is not None:
                xs = (x - 0.5) / 0.5
                if self.half:
                    xs = xs.half()
                s = self.siglip(pixel_values=xs).pooler_output.float()
                sg_out.append(F.normalize(s, dim=1).cpu().numpy())
        rn = np.concatenate(rn_out) if rn_out else np.zeros((0, 512), np.float32)
        sg = np.concatenate(sg_out) if sg_out else None
        return rn, sg


def t4_features(hist, rn, sg, use_sg):
    """Features of the supervised classifier: colour + ResNet (+ SigLIP when it was fitted with it)."""
    parts = [t1_features(hist, rn)]
    if use_sg:
        parts.append(sg)
    return np.concatenate(parts, axis=1).astype(np.float32)


def kmeans_p0(X, centers):
    """Soft membership of cluster 0 from distances to two centres."""
    d = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
    return d[:, 1] / (d[:, 0] + d[:, 1] + 1e-9)


class TeamModels:
    """Frozen team classifiers. predict() returns P(home) per crop for t1, t2, t3."""

    def __init__(self, arrays):
        self.a = {k: np.asarray(v) for k, v in arrays.items()}

    @classmethod
    def load(cls, path):
        with np.load(path) as z:
            return cls({k: z[k] for k in z.files})

    def save(self, path):
        np.savez(path, **self.a)

    def _home(self, p0, key):
        # p0 = P(cluster 0); <key>_home = which cluster is the home team
        return p0 if int(self.a[key + "_home"]) == 0 else 1.0 - p0

    def predict(self, hist, rn, sg):
        n = len(hist)
        out = {k: np.full(n, np.nan, np.float32) for k in ("t1", "t2", "t3", "t4", "r4")}
        if n == 0:
            return out
        out["t1"] = self._home(kmeans_p0(t1_features(hist, rn), self.a["t1_centers"]), "t1").astype(np.float32)
        out["t3"] = self._home(kmeans_p0(hist, self.a["t3_centers"]), "t3").astype(np.float32)
        if int(self.a.get("has_t2", 0)) and sg is not None:
            z = sg @ self.a["t2_w"] + float(self.a["t2_b"])
            p1 = 1.0 / (1.0 + np.exp(-z))          # P(cluster 1)
            out["t2"] = self._home(1.0 - p1, "t2").astype(np.float32)
        if "t4_w" in self.a:
            use_sg = bool(int(self.a["t4_use_sg"]))
            if sg is not None or not use_sg:
                z = t4_features(hist, rn, sg, use_sg) @ self.a["t4_w"].T + self.a["t4_b"]
                z = np.exp(z - z.max(1, keepdims=True))
                pr = z / z.sum(1, keepdims=True)                 # classes: 0 home, 1 away, 2 official
                out["t4"] = (pr[:, 0] / np.maximum(pr[:, 0] + pr[:, 1], 1e-9)).astype(np.float32)
                out["r4"] = pr[:, 2].astype(np.float32)
        return out


def save_json(obj, path):
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2)


def atomic_parquet(df, path):
    tmp = path + ".tmp"
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)


In [ ]:
from common import save_json

def find_one(name, required=True):
    hits = sorted(glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True))
    if not hits and required:
        raise FileNotFoundError(f"{name} not found under {DATA_DIR}: attach the dataset that holds it.")
    return hits[0] if hits else None

VIDEO = find_one("BAR-ATH_tactical.mp4")
ATD_TXT = find_one("BAR-ATH_tactical_atd.txt")
ATD_XML = find_one("BAR-ATH_tactical_atd.xml")
HOMO_CSV = find_one("BAR-ATH_tactical_homography.csv")
EVAL_DIR = os.path.join(WORK, "eval")
os.makedirs(EVAL_DIR, exist_ok=True)

RANGES = [(h, s, e if RUN_MODE == "full" else min(e, s + SMOKE_FRAMES - 1)) for h, s, e in HALVES]
EVAL = np.zeros(N_VIDEO, bool)
HALF_OF = np.zeros(N_VIDEO, np.int8)
for h, s, e in RANGES:
    EVAL[s:e + 1] = True
    HALF_OF[s:e + 1] = h
CALIB_FRAMES = np.zeros(0, int)        # the ATD cell below loads the scored frames only
if T5_TRACKS is None:
    T5_TRACKS = find_one(f"final_tracks_{RUN_MODE}.parquet", required=False) or find_one("final_tracks_full.parquet")
print("video:", VIDEO); print("tracks:", T5_TRACKS); print(f"mode={RUN_MODE}  scored frames={EVAL.sum():,}")

In [ ]:
import xml.etree.ElementTree as ET
from common import HOME, AWAY, NO_TEAM, ROLE_PLAYER, ROLE_GK, ROLE_REF, ROLE_NAME, TEAM_LETTER
L, W = PITCH

# ---- ATD metadata: slot -> team, home/away names, attack direction --------------------------
root = ET.parse(ATD_XML).getroot()
teams = [t for t in root.iter("Team")]
home_id, away_id = teams[0].get("id"), teams[1].get("id")       # first <Team> = home (FIFA/EPTS)
TEAM_NAMES = (teams[0].findtext("Name"), teams[1].findtext("Name"))
ATTACK_1ST = {}
for t, side in zip(teams[:2], (HOME, AWAY)):
    for pp in t.iter("ProviderParameter"):
        if pp.findtext("Name") == "attack_direction_first_half":
            ATTACK_1ST[side] = pp.findtext("Value")
player_team = {p.get("id"): p.get("teamId") for p in root.iter("Player")}
chan_player = {c.get("id"): c.get("playerId") for c in root.iter("PlayerChannel")}
spec = next(root.iter("DataFormatSpecification"))
slot_players = [chan_player[r.get("playerChannelId")] for r in spec.iter("PlayerChannelRef")
                if r.get("playerChannelId", "").endswith("_x")]
SLOT_TEAM = np.array([HOME if player_team.get(p) == home_id else AWAY if player_team.get(p) == away_id
                      else NO_TEAM for p in slot_players], np.int8)
N_SLOTS = len(SLOT_TEAM)
print(f"home={TEAM_NAMES[0]}  away={TEAM_NAMES[1]}  slots={N_SLOTS} "
      f"(H {np.sum(SLOT_TEAM == HOME)}, A {np.sum(SLOT_TEAM == AWAY)})  attack 1st half: {ATTACK_1ST}")

# ---- ATD positions (normalised 0-1) for the frames we score; video_frame = frame_id - 1 -------
t0 = time.time()
ATD = np.full((N_VIDEO, N_SLOTS, 2), np.nan, np.float32)
need = HALF_OF != 0
need[CALIB_FRAMES] = True
with open(ATD_TXT) as fh:
    for line in fh:
        fid, players, _ball = line.rstrip("\n").split(":")
        vf = int(fid) - 1
        if vf < 0 or vf >= N_VIDEO or not need[vf]:
            continue
        for i, p in enumerate(players.split(";")):
            if p[0] != "N":
                x, y, _ = p.split(",")
                ATD[vf, i, 0] = float(x)
                ATD[vf, i, 1] = float(y)
print(f"ATD parsed in {time.time() - t0:.0f}s; mean objects per scored frame: "
      f"{np.isfinite(ATD[EVAL, :, 0]).sum(1).mean():.1f}")

# ---- Homography: normalised pitch (0-1) -> pixels; inverse for pixels -> pitch --------------
hdf = pd.read_csv(HOMO_CSV)
H = hdf[[f"h{r}{c}" for r in range(3) for c in range(3)]].values.astype(np.float64).reshape(-1, 3, 3)
H_VALID = hdf["valid"].values.astype(bool)
H_INV = np.full_like(H, np.nan)
H_INV[H_VALID] = np.linalg.inv(H[H_VALID])
print(f"homography: {len(H):,} frames, valid {H_VALID.mean():.1%}, "
      f"median reprojection error in scored frames {np.nanmedian(hdf['reprojection_error'].values[EVAL]):.2f}")

def pix_to_pitch(frames, u, v):
    """Pixel feet points -> metres (x along the 105.3 m length, y across)."""
    Hi = H_INV[np.asarray(frames)]
    p = np.einsum("nij,nj->ni", Hi, np.stack([u, v, np.ones_like(u)], 1).astype(np.float64))
    return (p[:, 0] / p[:, 2] * L).astype(np.float32), (p[:, 1] / p[:, 2] * W).astype(np.float32)

def pitch_to_pix(frame, xn, yn):
    """Normalised pitch -> pixels for one frame."""
    p = H[frame] @ np.stack([xn, yn, np.ones_like(xn)]).astype(np.float64)
    return p[0] / p[2], p[1] / p[2]

def match(a, b, gate):
    """Hungarian match of two (n,2) point sets in metres; returns (ia, ib, dist) within gate."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0)
    d = np.linalg.norm(a[:, None, :] - b[None, :, :], axis=2)
    ia, ib = linear_sum_assignment(np.where(d > gate, 1e6, d))
    ok = d[ia, ib] <= gate
    return ia[ok], ib[ok], d[ia, ib][ok]

def atd_frame_m(f):
    """ATD positions in metres + slot team for one frame (finite slots only)."""
    xy = ATD[f]
    ok = np.isfinite(xy[:, 0])
    return xy[ok] * np.array([L, W], np.float32), SLOT_TEAM[ok], np.flatnonzero(ok)


## T5: strong team & role model (SigLIP2-so400m on whole-body crops, trained on ATD pseudo-labels)

**Why.** The team models above (T1–T4) are fitted on ~150 calibration frames of small **torso** crops
with colour histograms, ResNet18 and SigLIP-base. On the wide tactical camera, with half the pitch in
shade, that confuses the kits, and referees and goalkeepers leak into the teams.

**What T5 does**, on the final tracks (no new detection):
1. **Pseudo-labels from the ATD** (provided automatic data, team membership per slot) on `T5_TRAIN_FRAMES`
   frames spread over both halves: a box matched to an ATD player gets that player's team; matched and
   the deepest person in a penalty area with the detector's goalkeeper role → *home_gk / away_gk*;
   the detector's confident *referee* boxes not matched to any ATD object → *official*. No hand labels.
2. **Embeddings from SigLIP2-so400m** (~400M parameters, at 224 px: the crops are 40–90 px tall, so 384 px
   only cost 3× the time; falls back to 384 px, SigLIP-so400m, then SigLIP-base) on **whole-body** crops,
   one model copy per GPU, and a logistic regression over the 5 classes (≤ `T5_MAX_PER_CLASS` crops each).
3. **Held-out check:** trained on even 2-minute blocks and tested on odd ones (accuracy and confusion
   matrix printed), then refitted on everything.
4. **Per track segment** (`T5_SEG_SECONDS`): the class is the vote of `T5_CROPS_PER_SEG` crops. A segment
   whose median position is more than `T5_GK_MAX_DIST` m from both goal lines cannot be a goalkeeper (this
   settles goalkeeper vs referee when their kits look alike). Home / away → player of that team;
   home_gk / away_gk → goalkeeper of that team; official → referee.
5. **Checks** against the ATD (team agreement before and after), team sizes per frame (≤ 11), goalkeepers per
   team per frame (≤ 1), crop grids per class. Written to `final_tracks_<mode>_t5.parquet` (same columns, old
   values kept as `team_v1` / `role_v1`); the original file is not touched.

**Cost.** ~1 GB of weights (Internet on). On a full match: ~80k crops at inference and ≤ 15k at training,
on both T4s at 224 px; a progress line with the ETA is printed every minute. To re-run **only T5** on an
existing `final_tracks_full.parquet`, use `kaggle/kaggle_t5_teams.ipynb` (no detection, no tracking).

In [ ]:
# ---- T5 model and crop streaming ---------------------------------------------------------------
import torch.nn.functional as Fnn
from sklearn.linear_model import LogisticRegression

T5_CLASSES = ["home", "away", "home_gk", "away_gk", "official"]
C_HOME, C_AWAY, C_HGK, C_AGK, C_OFF = range(5)
T5_ROLE_TEAM = {C_HOME: ("player", "H"), C_AWAY: ("player", "A"), C_HGK: ("goalkeeper", "H"),
                C_AGK: ("goalkeeper", "A"), C_OFF: ("referee", "")}


def _t5_vec(out):
    """transformers < 5 returns a tensor from get_image_features, newer versions an output object."""
    if isinstance(out, torch.Tensor):
        return out
    return out.pooler_output if getattr(out, "pooler_output", None) is not None else out[0][:, 0]


class StrongEmbedder:
    """L2-normalised SigLIP(2) image embeddings of BGR crops, one model copy per GPU."""

    def __init__(self, names=T5_MODELS):
        from transformers import AutoModel, AutoImageProcessor
        n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
        self.devices = [f"cuda:{i}" for i in range(n_gpu if T5_MULTI_GPU else min(1, n_gpu))] or ["cpu"]
        self.dtype = torch.float16 if self.devices[0] != "cpu" else torch.float32
        last = None
        for n in names:
            try:
                self.models = [self._load(AutoModel, n, d) for d in self.devices]
                self.proc = AutoImageProcessor.from_pretrained(n)
                self.name = n
                break
            except Exception as exc:              # no internet / too old transformers -> next one
                last = exc
                print(f"T5: could not load {n}: {repr(exc)[:160]}")
        else:
            raise RuntimeError(f"T5: no SigLIP model could be loaded ({last!r})")
        import queue, threading
        self.free = queue.Queue()
        for i in range(len(self.devices)):
            self.free.put(i)
        print(f"T5 embedder: {self.name} on {self.devices}")

    def _load(self, AutoModel, name, dev):
        try:
            m = AutoModel.from_pretrained(name, torch_dtype=self.dtype, attn_implementation="sdpa")
        except Exception:
            m = AutoModel.from_pretrained(name, torch_dtype=self.dtype)
        return m.to(dev).eval()

    @torch.inference_mode()
    def embed_on(self, i, crops):
        rgb = [cv2.cvtColor(c, cv2.COLOR_BGR2RGB) for c in crops]
        px = self.proc(images=rgb, return_tensors="pt")["pixel_values"].to(self.devices[i], self.dtype)
        f = _t5_vec(self.models[i].get_image_features(pixel_values=px)).float()
        return Fnn.normalize(f, dim=-1).cpu().numpy()

    def __call__(self, crops):
        """Synchronous, on whichever GPU is free."""
        i = self.free.get()
        try:
            return self.embed_on(i, crops)
        finally:
            self.free.put(i)


def body_crop(frame, box, pad=T5_PAD):
    """Whole-body crop with a little context, or None for tiny / empty boxes."""
    x1, y1, x2, y2 = (float(v) for v in box)
    w, h = x2 - x1, y2 - y1
    if h < T5_MIN_BOX_H or w < 4:
        return None
    X1, Y1 = max(0, int(x1 - pad * w)), max(0, int(y1 - 0.5 * pad * h))
    X2, Y2 = min(frame.shape[1], int(x2 + pad * w)), min(frame.shape[0], int(y2 + 0.5 * pad * h))
    c = frame[Y1:Y2, X1:X2]
    return c if c.size else None


class FrameReader:
    """Frames in increasing order: grab forward for short hops, seek (checked) for long ones."""

    def __init__(self, path):
        self.cap, self.pos = cv2.VideoCapture(path), 0

    def read(self, f):
        f = int(f)
        if f < self.pos or f - self.pos > 300:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, f)
            self.pos = int(round(self.cap.get(cv2.CAP_PROP_POS_FRAMES)))
            if self.pos != f:
                self.cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, f - 250))
                self.pos = int(round(self.cap.get(cv2.CAP_PROP_POS_FRAMES)))
        while self.pos < f and self.cap.grab():
            self.pos += 1
        ok, img = self.cap.read()
        self.pos += 1
        return img if ok else None

    def close(self):
        self.cap.release()


def embed_requests(reqs, emb, desc):
    """reqs: list of (key, frame, (x1, y1, x2, y2)). Streams the video once in frame order; batches are
    embedded on the free GPU(s) in background threads while decoding continues.
    Returns (keys, embeddings float16) for the boxes that gave a crop."""
    from concurrent.futures import ThreadPoolExecutor
    reqs = sorted(reqs, key=lambda r: r[1])
    n_dev = len(getattr(emb, "devices", [None]))
    ex = ThreadPoolExecutor(max_workers=n_dev)
    pending, done_k, done_v = [], [], []
    bk, bc = [], []
    rd = FrameReader(VIDEO)
    t0 = last = time.time()
    n_emb = 0

    def submit():
        nonlocal bk, bc
        pending.append((bk, ex.submit(emb, bc)))
        bk, bc = [], []

    def collect(block):
        nonlocal n_emb
        while pending and (block or pending[0][1].done()):
            k, fut = pending.pop(0)
            v = fut.result().astype(np.float16)
            done_k.extend(k); done_v.append(v); n_emb += len(k)

    i = 0
    while i < len(reqs):
        f = reqs[i][1]
        j = i
        while j < len(reqs) and reqs[j][1] == f:
            j += 1
        img = rd.read(f)
        if img is not None:
            for key, _, box in reqs[i:j]:
                c = body_crop(img, box)
                if c is not None:
                    bk.append(key); bc.append(c)
            if len(bc) >= T5_BATCH:
                submit()
                if len(pending) >= 2 * n_dev:          # keep the GPUs fed, not the RAM
                    k, fut = pending.pop(0)
                    v = fut.result().astype(np.float16)
                    done_k.extend(k); done_v.append(v); n_emb += len(k)
                collect(False)
        i = j
        if time.time() - last > 60:
            el = time.time() - t0
            rate = j / max(el, 1e-6)
            print(f"  {desc}: {j:,}/{len(reqs):,} boxes read, {n_emb:,} embedded | {rate:.0f}/s | "
                  f"ETA {(len(reqs) - j) / max(rate, 1e-6) / 60:.0f} min", flush=True)
            last = time.time()
    if bc:
        submit()
    collect(True)
    ex.shutdown()
    rd.close()
    print(f"  {desc}: {n_emb:,} crops embedded in {time.time() - t0:.0f}s ({n_emb / max(time.time() - t0, 1e-6):.0f}/s)")
    dim = done_v[0].shape[1] if done_v else 1
    return done_k, (np.concatenate(done_v) if done_v else np.zeros((0, dim), np.float16))


def t5_deepest(g):
    """Bool per row of one frame: the person nearest the goal line inside each penalty area."""
    x, y = g["x_m"].to_numpy(float), g["y_m"].to_numpy(float)
    inb = ((x < 16.5) | (x > L - 16.5)) & (np.abs(y - W / 2) < 20.16) & np.isfinite(x)
    out = np.zeros(len(g), bool)
    for m, pick in ((inb & (x < L / 2), np.argmin), (inb & (x >= L / 2), np.argmax)):   # left goal, right goal
        if m.any():
            idx = np.flatnonzero(m)
            out[idx[pick(x[idx])]] = True
    return out

In [ ]:
# ---- T5 training: ATD pseudo-labels, held-out check, final fit ----------------------------------
T5_OK = False
if USE_T5:
    t5_path = T5_TRACKS or os.path.join(WORK, f"final_tracks_{RUN_MODE}.parquet")
    if not os.path.exists(t5_path):
        hit = find_one(os.path.basename(t5_path), required=False) if "find_one" in globals() else None
        t5_path = hit or t5_path
    if not os.path.exists(t5_path):
        print("T5 skipped: no tracks file", t5_path)
    else:
        T5_OK = True
if T5_OK:
    t0 = time.time()
    FT = pd.read_parquet(t5_path).reset_index(drop=True)
    if "EVAL" in globals():                              # smoke: only the frames this run scores
        FT = FT[EVAL[FT.frame.to_numpy()]].reset_index(drop=True)
    FT["box_h"] = (FT.y2 - FT.y1).astype(np.float32)
    print(f"T5 on {os.path.basename(t5_path)}: {len(FT):,} rows, {FT.track_id.nunique():,} tracks, "
          f"roles {FT.role.value_counts().to_dict()}")
    per = max(1, T5_TRAIN_FRAMES // max(1, len(RANGES)))
    train_frames = np.unique(np.concatenate([np.linspace(s + 25, e - 25, per).astype(int) for _, s, e in RANGES]))
    cand = FT[FT.frame.isin(train_frames) & (FT.box_h >= T5_MIN_BOX_H) & np.isfinite(FT.x_m)]
    lab_rows = []
    for f, g in cand.groupby("frame"):
        axy, ateam, _ = atd_frame_m(int(f))
        ia, ib, _d = match(g[["x_m", "y_m"]].to_numpy(np.float32), axy, gate=CALIB_MATCH_GATE)
        team = np.full(len(g), -1); team[ia] = ateam[ib]
        deep = t5_deepest(g)
        for j, r in enumerate(g.itertuples()):
            if team[j] in (HOME, AWAY):
                if deep[j] and r.role == "goalkeeper":
                    lab = C_HGK if team[j] == HOME else C_AGK
                elif not deep[j] and r.role == "player":
                    lab = C_HOME if team[j] == HOME else C_AWAY
                else:
                    continue
            elif team[j] < 0 and r.role == "referee" and r.conf >= T5_REF_CONF:
                lab = C_OFF
            else:
                continue
            lab_rows.append((r.Index, int(r.frame), (r.x1, r.y1, r.x2, r.y2), lab))
    y_all = np.array([r[3] for r in lab_rows])
    print("T5 pseudo-labels:", {T5_CLASSES[c]: int((y_all == c).sum()) for c in range(5)})
    rng_cap = np.random.default_rng(0)
    keep_i = np.concatenate([rng_cap.permutation(np.flatnonzero(y_all == c))[:T5_MAX_PER_CLASS] for c in range(5)])
    lab_rows = [lab_rows[i] for i in np.sort(keep_i)]
    y_all = y_all[np.sort(keep_i)]
    print(f"T5 training on {len(lab_rows):,} crops (at most {T5_MAX_PER_CLASS:,} per class)")
    if len(lab_rows) < 200 or len(np.unique(y_all)) < 2:
        print("T5 skipped: too few pseudo-labels (check the ATD alignment line of the calibration)")
        T5_OK = False
if T5_OK:
    T5_EMB = globals().get("T5_EMBEDDER") or StrongEmbedder()
    keys, X = embed_requests([(i, f, b) for i, f, b, _ in lab_rows], T5_EMB, "T5 train crops")
    ylab = dict((r[0], r[3]) for r in lab_rows)
    fr_of = dict((r[0], r[1]) for r in lab_rows)
    Y = np.array([ylab[k] for k in keys]); Fr = np.array([fr_of[k] for k in keys])
    Xf = X.astype(np.float32)
    block = (Fr // int(T5_BLOCK_S * FPS)) % 2
    T5_REPORT = {"embedder": getattr(T5_EMB, "name", type(T5_EMB).__name__), "train_crops": int(len(Y)),
                 "per_class": {T5_CLASSES[c]: int((Y == c).sum()) for c in range(5)}}
    tr, te = block == 0, block == 1
    if len(np.unique(Y[tr])) >= 2 and te.sum() >= 50:
        clf_h = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0).fit(Xf[tr], Y[tr])
        pred = clf_h.predict(Xf[te])
        acc = float((pred == Y[te]).mean())
        cm = pd.crosstab(pd.Series(Y[te], name="pseudo-label").map(dict(enumerate(T5_CLASSES))),
                         pd.Series(pred, name="predicted").map(dict(enumerate(T5_CLASSES))))
        print(f"T5 held-out (odd {T5_BLOCK_S}s blocks): accuracy {acc:.3f} on {te.sum()} crops")
        print(cm.to_string())
        T5_REPORT["heldout_accuracy"] = round(acc, 4)
        T5_REPORT["heldout_recall"] = {T5_CLASSES[c]: round(float((pred[Y[te] == c] == c).mean()), 3)
                                       for c in np.unique(Y[te])}
        if acc < 0.9:
            print("WARNING: held-out accuracy < 0.9 — look at the crop grids before trusting T5")
    T5_CLF = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0).fit(Xf, Y)
    print(f"T5 trained in {time.time() - t0:.0f}s")

In [ ]:
# ---- T5 inference per track segment, checks, output ------------------------------------------------
if T5_OK:
    t0 = time.time()
    FT["seg"] = ((FT.frame - FT.groupby("track_id").frame.transform("min")) // int(T5_SEG_SECONDS * FPS)).astype(np.int64)
    FT["segkey"] = FT.track_id.astype(np.int64) * 100000 + FT.seg
    big = FT[FT.box_h >= T5_MIN_BOX_H]
    reqs = []
    for sk, g in big.groupby("segkey"):
        pick = g.iloc[np.linspace(0, len(g) - 1, min(T5_CROPS_PER_SEG, len(g))).astype(int)]
        reqs += [(i, int(r.frame), (r.x1, r.y1, r.x2, r.y2)) for i, r in zip(pick.index, pick.itertuples())]
    print(f"T5: {FT.segkey.nunique():,} segments, {len(reqs):,} crops to embed")
    keys, X = embed_requests(reqs, T5_EMB, "T5 segment crops")
    cls_idx = list(T5_CLF.classes_)
    lp = T5_CLF.predict_log_proba(X.astype(np.float32))
    full = np.full((len(keys), 5), -30.0); full[:, cls_idx] = lp
    seg_of = FT.loc[keys, "segkey"].to_numpy()
    S = pd.DataFrame(full, columns=T5_CLASSES); S["segkey"] = seg_of
    mean_lp = S.groupby("segkey")[T5_CLASSES].mean()
    P = np.exp(mean_lp.to_numpy() - mean_lp.to_numpy().max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
    # position rule: far from both goal lines -> not a goalkeeper (renormalise without the GK classes)
    med_x = FT.groupby("segkey").x_m.median().reindex(mean_lp.index).to_numpy(float)
    far = np.isfinite(med_x) & (np.minimum(med_x, L - med_x) > T5_GK_MAX_DIST)
    gk_moved = int((far & np.isin(P.argmax(1), [C_HGK, C_AGK])).sum())
    P[far, C_HGK] = 0.0; P[far, C_AGK] = 0.0
    P /= P.sum(1, keepdims=True)
    print(f"T5 position rule: {gk_moved} segment(s) more than {T5_GK_MAX_DIST:.0f} m from both goals moved off goalkeeper")
    seg_cls = pd.Series(P.argmax(1), index=mean_lp.index)
    seg_conf = pd.Series(P.max(1), index=mean_lp.index)
    seg_phome = pd.Series(P[:, C_HOME] + P[:, C_HGK], index=mean_lp.index)

    FT["role_v1"], FT["team_v1"] = FT.role, FT.team
    has = FT.segkey.isin(seg_cls.index)
    c = FT.loc[has, "segkey"].map(seg_cls).astype(int)
    FT.loc[has, "role"] = c.map(lambda k: T5_ROLE_TEAM[k][0])
    FT.loc[has, "team"] = c.map(lambda k: T5_ROLE_TEAM[k][1])
    FT["t5_class"] = ""
    FT.loc[has, "t5_class"] = c.map(dict(enumerate(T5_CLASSES)))
    FT["t5_conf"] = FT.segkey.map(seg_conf).astype(np.float32)
    FT.loc[has, "team_p_home"] = FT.loc[has, "segkey"].map(seg_phome).astype(np.float32)
    print(f"T5 assigned {has.mean():.1%} of rows ({(~has).sum():,} rows kept their v1 labels: every box of the "
          f"segment smaller than {T5_MIN_BOX_H}px) | {time.time() - t0:.0f}s")

    # ---- checks: ATD agreement before / after, team sizes, goalkeepers, officials -------------------
    def atd_agreement(team_col, frames):
        ok = n = 0
        for f in frames:
            g = FT[(FT.frame == f) & FT[team_col].isin(["H", "A"]) & np.isfinite(FT.x_m)]
            if not len(g):
                continue
            axy, ateam, _ = atd_frame_m(int(f))
            ia, ib, _d = match(g[["x_m", "y_m"]].to_numpy(np.float32), axy, gate=CALIB_MATCH_GATE)
            want = np.where(ateam[ib] == HOME, "H", "A")
            ok += int((g[team_col].to_numpy()[ia] == want).sum()); n += len(ia)
        return ok / max(n, 1), n
    eval_frames = np.unique(np.concatenate([np.arange(s, e + 1, 125) for _, s, e in RANGES]))
    eval_frames = np.setdiff1d(eval_frames, train_frames)
    a_old, n_old = atd_agreement("team_v1", eval_frames)
    a_new, n_new = atd_agreement("team", eval_frames)
    cnt = FT[FT.frame.isin(eval_frames)].groupby(["frame", "team", "role"]).size().unstack(["team", "role"], fill_value=0)
    def share_le(cols, k):
        sub = [c for c in cnt.columns if c in cols]
        return float((cnt[sub].sum(1) <= k).mean()) if sub else float("nan")
    checks = {
        "atd_team_agreement_v1": round(a_old, 4), "atd_team_agreement_t5": round(a_new, 4), "atd_matched_boxes": n_new,
        "frames_home_le_11": round(share_le([("H", "player"), ("H", "goalkeeper")], 11), 4),
        "frames_away_le_11": round(share_le([("A", "player"), ("A", "goalkeeper")], 11), 4),
        "frames_home_gk_le_1": round(share_le([("H", "goalkeeper")], 1), 4),
        "frames_away_gk_le_1": round(share_le([("A", "goalkeeper")], 1), 4),
        "officials_per_frame": round(float(cnt[[c for c in cnt.columns if c[1] == "referee"]].sum(1).mean()), 2)
        if any(c[1] == "referee" for c in cnt.columns) else 0.0,
        "rows_team_changed": round(float((FT.team != FT.team_v1).mean()), 4),
        "rows_role_changed": round(float((FT.role != FT.role_v1).mean()), 4),
    }
    T5_REPORT.update(checks=checks, segments=int(len(seg_cls)), gk_moved_by_position=gk_moved, rows_assigned=round(float(has.mean()), 4),
                     classes_assigned={k: int(v) for k, v in FT.loc[has, "t5_class"].value_counts().items()})
    print(json.dumps(checks, indent=1))
    if a_new < a_old:
        print("WARNING: T5 agrees LESS with the ATD teams than v1 — keep final_tracks_<mode>.parquet")

    # ---- crop grid per class (high-confidence segments) --------------------------------------------
    rng = np.random.default_rng(0)
    rd = FrameReader(VIDEO)
    fig, axes = plt.subplots(5, 12, figsize=(12, 6.2))
    for ci, name in enumerate(T5_CLASSES):
        segs = seg_cls.index[(seg_cls == ci).to_numpy() & (seg_conf >= 0.6).to_numpy()]
        segs = rng.permutation(segs)[:12]
        rows_ = [FT[(FT.segkey == s) & (FT.box_h >= T5_MIN_BOX_H)].iloc[0] for s in segs]
        rows_ = sorted(rows_, key=lambda r: r.frame)
        for j in range(12):
            ax = axes[ci, j]; ax.axis("off")
            if j < len(rows_):
                r = rows_[j]; img = rd.read(r.frame)
                cr = body_crop(img, (r.x1, r.y1, r.x2, r.y2)) if img is not None else None
                if cr is not None:
                    ax.imshow(cv2.cvtColor(cv2.resize(cr, (40, 80)), cv2.COLOR_BGR2RGB))
        axes[ci, 0].set_title(f"{name}: {int((seg_cls == ci).sum())} segments", fontsize=8, loc="left")
    rd.close()
    plt.tight_layout(); plt.savefig(os.path.join(EVAL_DIR, f"t5_classes_{RUN_MODE}.png"), dpi=110); plt.show()

    out_cols = ["frame", "half", "track_id", "role", "team", "conf", "x1", "y1", "x2", "y2", "x_m", "y_m",
                "team_p_home", "t5_class", "t5_conf", "role_v1", "team_v1"]
    out_path = os.path.join(WORK, f"final_tracks_{RUN_MODE}_t5.parquet")
    FT[out_cols].to_parquet(out_path, index=False)
    save_json(T5_REPORT, os.path.join(EVAL_DIR, f"t5_report_{RUN_MODE}.json"))
    print("T5 written:", out_path)
    del T5_EMB
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()